# 🏙️ Multi-City Air Quality Comparison

Compare annual NO₂ trends across **Paris, London, Amsterdam and Berlin** for 2023.  
We use `aggregate="area_monthly"` to get a single area-averaged value per city per month  
— keeping query costs low (1 tile per city per month).

**Requirements**
```
pip install jiskta matplotlib seaborn pandas numpy
```

In [ ]:
# Install the SDK from the private GitHub repo.
# Generate a read-only token at: https://github.com/settings/tokens/new
#   → select scope: Contents (read-only)
# Then run (replace TOKEN with your actual token):
# !pip install "git+https://TOKEN@github.com/fvsever/jiskta-python.git#egg=jiskta[pandas]" matplotlib seaborn numpy

In [ ]:
from jiskta import JisktaClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

API_KEY = "sk_live_YOUR_API_KEY"
client = JisktaClient(api_key=API_KEY)

# City bounding boxes  (lat_min, lat_max, lon_min, lon_max)
CITIES = {
    "Paris":     {"lat": (48.7, 49.0),  "lon": (2.2,  2.5)},
    "London":    {"lat": (51.4, 51.6),  "lon": (-0.2, 0.1)},
    "Amsterdam": {"lat": (52.3, 52.5),  "lon": (4.8,  5.0)},
    "Berlin":    {"lat": (52.4, 52.6),  "lon": (13.3, 13.5)},
}

## 1 — Fetch monthly area averages for each city

In [ ]:
frames = []

for city, bbox in CITIES.items():
    df = client.query(
        lat=bbox["lat"],
        lon=bbox["lon"],
        start="2023-01",
        end="2023-12",
        variables=["no2"],
        aggregate="area_monthly",
    )
    df["city"] = city
    frames.append(df)
    print(f"  {city}: {len(df)} months fetched")

data = pd.concat(frames, ignore_index=True)
data["year_month"] = pd.to_datetime(data["year_month"])
data.head(8)

## 2 — Monthly NO₂ line chart

In [ ]:
palette = {"Paris": "#1a6bbf", "London": "#e05252", "Amsterdam": "#27ae60", "Berlin": "#8e44ad"}

fig, ax = plt.subplots(figsize=(12, 5))
for city, grp in data.groupby("city"):
    grp = grp.sort_values("year_month")
    ax.plot(grp["year_month"], grp["no2_mean"],
            marker="o", linewidth=2.2, markersize=6,
            color=palette[city], label=city)

ax.axhline(10, color="#c0392b", linewidth=1, linestyle="--", label="WHO annual guideline (10 µg/m³)")
ax.set_title("Monthly mean NO₂ — Paris · London · Amsterdam · Berlin (2023)", fontsize=14, pad=12)
ax.set_ylabel("NO₂  (µg/m³)")
ax.set_xlabel("")
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 3 — Annual ranking (bar chart)

In [ ]:
annual = data.groupby("city")["no2_mean"].mean().sort_values(ascending=False).reset_index()
annual.columns = ["City", "Annual mean NO₂ (µg/m³)"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(annual["City"], annual["Annual mean NO₂ (µg/m³)"],
               color=[palette[c] for c in annual["City"]], alpha=0.85)
ax.axvline(10, color="#c0392b", linewidth=1.2, linestyle="--", label="WHO annual guideline (10 µg/m³)")

for bar, val in zip(bars, annual["Annual mean NO₂ (µg/m³)"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontsize=10)

ax.set_xlabel("Annual mean NO₂  (µg/m³)")
ax.set_title("Annual average NO₂ by city — 2023", fontsize=13, pad=10)
ax.legend()
ax.invert_yaxis()
fig.tight_layout()
plt.show()

annual

## 4 — Heatmap: month × city

In [ ]:
pivot = data.pivot_table(index="city", columns=data["year_month"].dt.strftime("%b"),
                         values="no2_mean", aggfunc="mean")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot = pivot[month_order]

fig, ax = plt.subplots(figsize=(12, 3.5))
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "NO₂  (µg/m³)"})
ax.set_title("Monthly mean NO₂  (µg/m³) — city comparison, 2023", fontsize=13, pad=10)
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

## 5 — Extend: add PM2.5 and ERA5 wind speed

You can enrich the analysis by adding more variables to the same query:

```python
df = client.query(
    lat=bbox["lat"], lon=bbox["lon"],
    start="2023-01", end="2023-12",
    variables=["no2", "pm2p5", "u10", "v10"],   # air quality + ERA5 wind
    aggregate="area_monthly",
)
```

`u10` and `v10` are the eastward and northward 10-m wind components (m/s) from ERA5.

**See [jiskta.com/docs](https://jiskta.com/docs) for all available variables.**